# Automating Workflows with Schedulers (CRON and Python `schedule`)


## 1. What is Workflow Automation?

Workflow automation means executing tasks automatically at predefined times or intervals.

Examples:
- Run ETL every day at 2 AM
- Backup database every hour
- Monitor website every 30 seconds
- Train ML model weekly

## 2. CRON Scheduler (Linux/Unix)

**CRON** is a system-level scheduler that runs commands at specified times.

### CRON Syntax:
```
* * * * * command
| | | | |
| | | | └── Day of week (0–7)
| | | └──── Month (1–12)
| | └────── Day of month (1–31)
| └──────── Hour (0–23)
└────────── Minute (0–59)
```

### CRON Examples:
- Run script every day at 2:30 AM:
```
30 2 * * * python etl.py
```
- Run every 5 minutes:
```
*/5 * * * * python job.py
```
- Run every Monday at 9 AM:
```
0 9 * * 1 python report.py
```

### CRON Commands
- Edit crontab: `crontab -e`
- List jobs: `crontab -l`
- Remove all jobs: `crontab -r`

## 3. Python `schedule` Library

`schedule` is a lightweight Python library to schedule tasks programmatically.

Python schedule is a lightweight library for scheduling Python functions to run at regular intervals or specific times. 

Unlike CRON, it runs inside a Python process, giving flexibility for conditional logic, threading, and logging.

Install schedule if not installed
`pip install schedule`

### Basic Example

In [1]:
import schedule
import time

def job():
    print("Task running...")

schedule.every(5).seconds.do(job)

start_time = time.time()
while time.time() - start_time < 20:  # run for 20 seconds
    schedule.run_pending()
    time.sleep(1)

Task running...
Task running...
Task running...


### Interval and Time-based Scheduling

Python schedule allows you to run functions at fixed intervals (every N seconds/minutes) or at specific times of the day.

#### Interval-Based Scheduling : 
Run a job repeatedly after a fixed interval (e.g., every 5 seconds, every 2 minutes).

Syntax:

schedule.every(interval).seconds.do(job)
schedule.every(interval).minutes.do(job)
schedule.every(interval).hours.do(job)
schedule.every(interval).days.do(job)

#### Time-Based Scheduling  :
Run a job at a specific time of the day or week (e.g., daily at 02:30 AM, every Monday at 9:00 AM).
Syntax:

schedule.every().day.at("HH:MM").do(job)
schedule.every().monday.at("HH:MM").do(job)

In [2]:
def task():
    print("Task executed")

# Interval based
schedule.every(10).seconds.do(task)
schedule.every(2).minutes.do(task)
schedule.every().hour.do(task)

# Time based
schedule.every().day.at("14:00").do(task)
schedule.every().monday.do(task)


Every 1 week do task() (last run: [never], next run: 2026-03-23 11:05:06)

### Tagging Jobs and Cancelling

In [3]:
schedule.every(15).seconds.do(task).tag('etl')
# Clear all ETL tagged jobs
# schedule.clear('etl')

Every 15 seconds do task() (last run: [never], next run: 2026-03-21 11:05:21)

## 4. Graceful Shutdown Pattern

A graceful shutdown is when a program stops accepting new tasks, finishes currently running tasks, and exits cleanly without leaving resources (threads, files, database connections) in an inconsistent state.

In [4]:
import threading
import logging
import signal
from datetime import datetime

logging.basicConfig(level=logging.INFO)

stop_event = threading.Event()

def signal_handler(sig, frame):
    logging.info("Shutdown signal received")
    stop_event.set()

signal.signal(signal.SIGINT, signal_handler)
signal.signal(signal.SIGTERM, signal_handler)

def ETL_job():
    logging.info(f"ETL started at {datetime.now()}")
    time.sleep(4)
    logging.info("ETL finished")

schedule.every(5).seconds.do(ETL_job)

def run_scheduler():
    while not stop_event.is_set():
        schedule.run_pending()
        time.sleep(1)
    logging.info("Scheduler stopped")

scheduler_thread = threading.Thread(target=run_scheduler)
scheduler_thread.start()

# Main thread doing other work
try:
    for i in range(1, 6):
        print(f"Main thread working... {i}")
        time.sleep(3)
finally:
    stop_event.set()
    scheduler_thread.join()
    logging.info("Program exited cleanly")

Task running...
Main thread working... 1
Main thread working... 2


INFO:root:ETL started at 2026-03-21 11:05:11.610046


Main thread working... 3


INFO:root:ETL finished


Main thread working... 4
Task running...
Task executed
Main thread working... 5


INFO:root:ETL started at 2026-03-21 11:05:20.612716
INFO:root:ETL finished


Task running...


INFO:root:Scheduler stopped
INFO:root:Program exited cleanly


## 5. CRON vs schedule


| **Aspect**                | **CRON**                                                                                 | **Python `schedule`**                                                                                        |
| ------------------------- | ---------------------------------------------------------------------------------------- | ---------------------------------------------------------------------------------------- |
| **Definition**            | CRON is an OS-level scheduler that runs commands or scripts at predefined times.         | `schedule` is a Python library that runs Python functions at defined intervals or specific times.            |
| **Execution Level**       | Runs at the operating system level independently of any programming language.            | Runs inside a Python process; the script must be running for tasks to execute.                               |
| **Platform**              | Unix/Linux only (Windows uses Task Scheduler instead).                                   | Cross-platform: works on Linux, Windows, macOS.                                                              |
| **Syntax / Setup**        | Configured via crontab with 5 fields: minute, hour, day, month, weekday.                 | Configured in Python code: `schedule.every().seconds.do(job)` or `schedule.every().day.at("02:00").do(job)`. |
| **Reliability**           | Very high: jobs run as long as the system is on, even if other programs fail.            | Medium: requires Python script running; if script crashes, jobs stop.                                        |
| **Flexibility**           | Limited logic: runs commands at fixed times, no conditional execution.                   | Highly flexible: can include Python logic, loops, dependencies, retries, and error handling.                 |
| **Logging**               | Must redirect stdout/stderr to file or use email alerts; manual setup.                   | Easy with Python logging module; logs can go to console or file.                                             |
| **Parallel Execution**    | Runs one command per schedule; for parallel jobs, multiple scripts are needed.           | Easy with threads or async; multiple jobs can run in parallel in the same script.                            |
| **Typical Use Cases**     | Production system-level automation: backups, ETL triggers, report generation.            | Application-level automation: ETL tasks, monitoring, sending notifications, testing workflows.               |

